In [13]:
from dotenv import load_dotenv
from langchain_teddynote import logging

load_dotenv()
logging.langsmith("CH02-Prompt")
'''
import os


def langsmith(project_name=None, set_enable=True):

    if set_enable:
        langchain_key = os.environ.get("LANGCHAIN_API_KEY", "")
        langsmith_key = os.environ.get("LANGSMITH_API_KEY", "")

        # 더 긴 API 키 선택
        if len(langchain_key.strip()) >= len(langsmith_key.strip()):
            result = langchain_key
        else:
            result = langsmith_key

        if result.strip() == "":
            print(
                "LangChain/LangSmith API Key가 설정되지 않았습니다. 참고: https://wikidocs.net/250954"
            )
            return

        os.environ["LANGSMITH_ENDPOINT"] = (
            "https://api.smith.langchain.com"  # LangSmith API 엔드포인트
        )
        os.environ["LANGSMITH_TRACING"] = "true"  # true: 활성화
        os.environ["LANGSMITH_PROJECT"] = project_name  # 프로젝트명
        print(f"LangSmith 추적을 시작합니다.\n[프로젝트명]\n{project_name}")
    else:
        os.environ["LANGSMITH_TRACING"] = "false"  # false: 비활성화
        print("LangSmith 추적을 하지 않습니다.")
'''

LangSmith 추적을 시작합니다.
[프로젝트명]
CH02-Prompt


'\nimport os\n\n\ndef langsmith(project_name=None, set_enable=True):\n\n    if set_enable:\n        langchain_key = os.environ.get("LANGCHAIN_API_KEY", "")\n        langsmith_key = os.environ.get("LANGSMITH_API_KEY", "")\n\n        # 더 긴 API 키 선택\n        if len(langchain_key.strip()) >= len(langsmith_key.strip()):\n            result = langchain_key\n        else:\n            result = langsmith_key\n\n        if result.strip() == "":\n            print(\n                "LangChain/LangSmith API Key가 설정되지 않았습니다. 참고: https://wikidocs.net/250954"\n            )\n            return\n\n        os.environ["LANGSMITH_ENDPOINT"] = (\n            "https://api.smith.langchain.com"  # LangSmith API 엔드포인트\n        )\n        os.environ["LANGSMITH_TRACING"] = "true"  # true: 활성화\n        os.environ["LANGSMITH_PROJECT"] = project_name  # 프로젝트명\n        print(f"LangSmith 추적을 시작합니다.\n[프로젝트명]\n{project_name}")\n    else:\n        os.environ["LANGSMITH_TRACING"] = "false"  # false: 비활성화\n        print

In [ ]:
from langchain_openai import ChatOpenAI
llm = ChatOpenAI(model="gpt-4o-mini",max_completion_tokens=2048)

In [2]:
from langchain_core.prompts import PromptTemplate

template = "{country}의 수도는 어디인가요?"

In [8]:
#prompt = PromptTemplate.from_template(template)
prompt = PromptTemplate(
    template=template,
    input_variables=["country"],
    )
prompt.format(country="대한민국")

'대한민국의 수도는 어디인가요?'

In [11]:
chain = prompt | llm 
chain.invoke("대한민국").content

'대한민국의 수도는 서울입니다.'

In [17]:
template = "{country_1}과 {country_2}의 수도는 각각 어디인가요?"
prompt = PromptTemplate(
    template=template,
    input_variables=["country_1"],
    partial_variables={
        "country_2" : "미국",
    }
)

prompt

PromptTemplate(input_variables=['country_1'], input_types={}, partial_variables={'country_2': '미국'}, template='{country_1}과 {country_2}의 수도는 각각 어디인가요?')

In [22]:
prompt.format(country_1="대한민국")

'대한민국과 미국의 수도는 각각 어디인가요?'

In [20]:
prompt_partial = prompt.partial(country_2="캐나다")
prompt_partial

PromptTemplate(input_variables=['country_1'], input_types={}, partial_variables={'country_2': '캐나다'}, template='{country_1}과 {country_2}의 수도는 각각 어디인가요?')

In [23]:
prompt_partial.format(country_1="대한민국")

'대한민국과 캐나다의 수도는 각각 어디인가요?'

In [ ]:
chain = prompt_partial | llm
#이거는 왜 동작하지 ? 답변은 한국의 수도는 으로 나오던데... 
# chain.invoke().content  

chain.invoke("대한민국").content

'대한민국의 수도는 서울이며, 캐나다의 수도는 오타와입니다.'

In [26]:
chain.invoke({"country_1" : "대한민국", "country_2" : "호주"}).content

'대한민국의 수도는 서울이고 호주의 수도는 캔버라입니다.'

In [29]:
from datetime import datetime

def get_today():
    return datetime.now().strftime("%B %d")
print(get_today())

September 15


In [31]:
prompt = PromptTemplate(
    template="오늘의 날짜는 {today}입니다. 오늘이 생일인 유명인 {n}명을 나열해 주세요. 생년월일을 표기해주세요.",
    input_variables=["n"],
    partial_variables={
        "today": get_today
    },
)

In [32]:
prompt.format(n=3)

'오늘의 날짜는 September 15입니다. 오늘이 생일인 유명인 3명을 나열해 주세요. 생년월일을 표기해주세요.'

In [33]:
chain = prompt | llm 
print(chain.invoke(3).content)

1. Prince Harry (1984년 9월 15일)
2. Tommy Lee Jones (1946년 9월 15일)
3. Tom Hardy (1977년 9월 15일)


In [34]:
print(chain.invoke({"today" : "April 24", "n":3}).content)

1. Kelly Clarkson - 1982년 4월 24일
2. Barbra Streisand - 1942년 4월 24일
3. Sachin Tendulkar - 1973년 4월 24일
